# En este cuaderno se explorará el dataset de Hugging Face de los autores Calizaya y Santos, ya que ello fue lo utilizado por ellos según sus Notebooks de su trabajo: https://huggingface.co/datasets/pyupeu/social-media-peruvian-sentiment

In [1]:
!pip install -q datasets pandas


In [2]:
import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_colwidth", 300)


Cargar dataset desde Hugging Face

In [3]:
ds = load_dataset("pyupeu/social-media-peruvian-sentiment")
ds


README.md:   0%|          | 0.00/857 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  990kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  306kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9336 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2918 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2335 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 9336
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 2918
    })
    test: Dataset({
        features: ['text', 'label', 'label_name'],
        num_rows: 2335
    })
})

Revisar qué splits trae el dataset

In [4]:
dfs = {split_name: ds[split_name].to_pandas() for split_name in ds.keys()}

for nombre, df in dfs.items():
    print(f"{nombre.upper():<12} -> filas: {df.shape[0]:>6} | columnas: {df.shape[1]}")


TRAIN        -> filas:   9336 | columnas: 3
VALIDATION   -> filas:   2918 | columnas: 3
TEST         -> filas:   2335 | columnas: 3


Columnas y muestra de cada split

In [5]:
for nombre, df in dfs.items():
    print(f"\n{nombre.upper()}")
    print("Columnas:", df.columns.tolist())

# Verificar si todos los splits tienen exactamente las mismas columnas
columnas_por_split = {nombre: set(df.columns) for nombre, df in dfs.items()}
nombres = list(columnas_por_split.keys())
todas_iguales = all(columnas_por_split[nombres[0]] == columnas_por_split[n] for n in nombres[1:])
print("\n¿Todas las columnas son iguales entre splits?:", todas_iguales)
if not todas_iguales:
    for n in nombres:
        print(n, "->", columnas_por_split[n])


TRAIN
Columnas: ['text', 'label', 'label_name']

VALIDATION
Columnas: ['text', 'label', 'label_name']

TEST
Columnas: ['text', 'label', 'label_name']

¿Todas las columnas son iguales entre splits?: True


In [6]:
list(dfs.values())[0].head(10)


,text,label,label_name
0,"Salió para Aser farándula ese chato, solo quiere llamar la atención 😂 q trabaje bago , aragán",0,negative
1,Si así eres tu también!!! ... Cuando llegaste a Cancun al hotel donde trabajaba.. Y la netta casi todos son bien mamones.. Y otros mamones y piojos jajaj pero bueno en sus videos son otra cosa👎🏻,0,negative
2,Nunca se tiene pocos zapatos... 🤣 🤣 🤣 🤣 Tengo algunos tacones y botas que aun no los e usado y sigo comprando mas.,1,neutral
3,Queen Reigne te 😂😂😂 sorry na,1,neutral
4,En media hora comienza el toque de queda dijo,1,neutral
5,Ahuevo ahora le podré gustar a mi amiga Otaku 😎👍,2,positive
6,en él norte dicen si vas a trujillo y no fuiste a huanchaco a que fuiste a trujillo ....😁 aplicara para la nueva Palomino?? 😏,1,neutral
7,"Los KONGRESISTAS del FUJIMONTESINISMO son los Menos indicado de hablar de Corrupción.. Es normal que los KONGRESISTA Asusan a la población como lo hace la Moyano la BARBARAN , Yarrow, Montoya , Cueto ,Nano Guerra, el mismo CHIABRA etc etc. Ya ps tan ingenuo no somos. YEGO el DESPERTAR de las ...",1,neutral
8,🤣🤣🤣🤣 Manolo yo tambien estoy chihuan..,1,neutral
9,Kevin seria bueno que cumplan estas reglas 😅😆,1,neutral


In [9]:
total_hf = sum(df.shape[0] for df in dfs.values())
print("Total de registros en el dataset de Hugging Face:", total_hf)

referencias = {
    "Paper (Calizaya et al., texto)": 11276
}

print("\nComparación contra otras fuentes conocidas:")
for fuente, cantidad in referencias.items():
    diferencia = total_hf - cantidad
    print(f"  {fuente:<55} {cantidad:>6}  (diferencia vs HF: {diferencia:+d})")

Total de registros en el dataset de Hugging Face: 14589

Comparación contra otras fuentes conocidas:
  Paper (Calizaya et al., texto)                           11276  (diferencia vs HF: +3313)


El dataset de Hugging Face tiene 3313 registros más que los declarados en el trabajo de Calizaya y Santos.

## Distribución de clases por split

In [10]:
# Ajustar el nombre de la columna de etiqueta si es distinto (p. ej. 'label', 'label_name', 'polarity')
COLUMNA_LABEL_CANDIDATAS = ["label_name", "label"]

columna_label = None
for candidata in COLUMNA_LABEL_CANDIDATAS:
    if candidata in list(dfs.values())[0].columns:
        columna_label = candidata
        break

print("Columna de etiqueta detectada:", columna_label)

for nombre, df in dfs.items():
    print(f"\n{nombre.upper()}")
    print(df[columna_label].value_counts())
    print((df[columna_label].value_counts(normalize=True) * 100).round(2))

Columna de etiqueta detectada: label_name

TRAIN
label_name
negative    4265
positive    3033
neutral     2038
Name: count, dtype: int64
label_name
negative    45.68
positive    32.49
neutral     21.83
Name: proportion, dtype: float64

VALIDATION
label_name
negative    1333
positive     948
neutral      637
Name: count, dtype: int64
label_name
negative    45.68
positive    32.49
neutral     21.83
Name: proportion, dtype: float64

TEST
label_name
negative    1066
positive     759
neutral      510
Name: count, dtype: int64
label_name
negative    45.65
positive    32.51
neutral     21.84
Name: proportion, dtype: float64


## Duplicados dentro de cada split

In [13]:
# Tomamos la columna text
COLUMNA_TEXTO_CANDIDATAS = ["text", "text_original", "tokenized_text", "comentario", "comment"]

columnas_texto_presentes = [c for c in COLUMNA_TEXTO_CANDIDATAS if c in list(dfs.values())[0].columns]
print("Columnas de texto presentes:", columnas_texto_presentes)

muestra = list(dfs.values())[0][columnas_texto_presentes + ([columna_label] if columna_label else [])].sample(10, random_state=42)
muestra

Columnas de texto presentes: ['text']


,text,label_name
4936,Hola Luciano un gusto saludarte aca en 🇨🇱 igual lo podemos comer como lo has preparado pero igual se prepara mas agua con chancaca y Maicena para echar adentro los picarones que se remojen unos minutos y luego comer todo junto esquisito. Un saludo grande desde'Chile'🇨🇱,neutral
7553,"Ni sueñes que serás presidente , ni tu ni tu paisano Castillo son buenas personas capaces de representar a un país, mucho menos al Perú 🇵🇪. Charlatanes trafaciosos que solo busca robar nuestro dinero de todos los buenos Ciudadanos que nos sacamos la mierda día a día para sacar adel...",negative
3457,"Pero ese vídeo es muy corto, haznos uno sobre tipos de carpa, tipos de sacos de dormir... 😊",positive
6250,que palta 😂😂😂pongan los nombres de esos ingenuos 😂,negative
3455,Todo por el rating 🤣🤣 y los aliados del dióxido de cloro que lo miran 🤣🤣🤣🤦,negative
4862,"Pobre mujer, hasta dónde se ha degradado. Después de enseñar hasta la molleja, por dinero... ahora busca exposición sólo criticando a la que por mucho la ha superado en cantidad, calidad y variedad de contenido. Qué bajo que recurra al macheteo para figurar. 🤦🏻‍♂️.... en fin, lío de putas 🤷🏻‍♂️🤷...",negative
457,Una huachafa fea,negative
6378,Siiiiii Toby!!... Cómo olvidar esa chamba cuando Luis Felipe Amaya Broncano era pareja de Lenny Paul Andres Fuertes y se metían su respectiva Half-Life con Gabito Traguncitus Jodiduz 😂 Pero cabinero que se respeta!!... Le metía reja a las clientas 🤣... Que recuerdos de ese baño tantos momentos v...,positive
6533,Muy 👍🦸‍♂️👍🦸‍♀️el paseo felicitaciones en invierno El Lucumo se convierte en un paraíso verde y florido muy bello,positive
2882,Gerardo Pe' mano ese caldito te levanta hast prodia hacerle hueco ala pared jaja saludos el mejor caldo en la vicky 😎,positive


In [15]:
columna_texto_principal = columnas_texto_presentes[0]
print("Usando como columna de texto principal para duplicados:", columna_texto_principal)

for nombre, df in dfs.items():
    print(f"\n{nombre.upper()}")
    for c in columnas_texto_presentes:
        print(f"  Duplicados en '{c}':", df[c].duplicated().sum())

Usando como columna de texto principal para duplicados: text

TRAIN
  Duplicados en 'text': 26

VALIDATION
  Duplicados en 'text': 0

TEST
  Duplicados en 'text': 0


## Duplicados entre splits

In [16]:
df_all = pd.concat(
    [df.assign(split=nombre) for nombre, df in dfs.items()],
    ignore_index=True
)

duplicados_entre_splits = (
    df_all[df_all[columna_texto_principal].duplicated(keep=False)]
    .sort_values(columna_texto_principal)
)

print("Filas involucradas en duplicados por texto (entre todos los splits):", len(duplicados_entre_splits))

duplicados_entre_splits[[columna_texto_principal, columna_label, "split"]].head(30)


Filas involucradas en duplicados por texto (entre todos los splits): 83


,text,label_name,split
2369,Acuérdate que todas son cachables,neutral,train
12454,Acuérdate que todas son cachables,neutral,test
9646,"Ahora, olvídese del asunto y déjelo a mi cargo",neutral,validation
7966,"Ahora, olvídese del asunto y déjelo a mi cargo",negative,train
7837,Ahorita vengo,neutral,train
5888,Ahorita vengo,neutral,train
2645,Aliancista y la conchatumadre,negative,train
6157,Aliancista y la conchatumadre,negative,train
5473,Creeras que te miento pero llevo 17 añitos sin comerme un delicious seco com frejoles!!! 😋😋😋🤤🤤🤤,neutral,train
2508,Creeras que te miento pero llevo 17 añitos sin comerme un delicious seco com frejoles!!! 😋😋😋🤤🤤🤤,neutral,train


## Etiquetas contradictorias (mismo texto, distinta etiqueta)

In [17]:
revision_duplicados = (
    df_all[df_all[columna_texto_principal].duplicated(keep=False)]
    .groupby(columna_texto_principal)
    .agg(
        cantidad=(columna_texto_principal, "count"),
        etiquetas=(columna_label, lambda x: sorted(set(x))),
        splits=("split", lambda x: sorted(set(x)))
    )
    .reset_index()
)

revision_duplicados["num_etiquetas"] = revision_duplicados["etiquetas"].apply(len)
revision_duplicados["num_splits"] = revision_duplicados["splits"].apply(len)

print("Textos duplicados únicos:", len(revision_duplicados))
print("Textos duplicados con etiquetas distintas (contradictorios):", (revision_duplicados["num_etiquetas"] > 1).sum())
print("Textos duplicados presentes en más de un split:", (revision_duplicados["num_splits"] > 1).sum())

revision_duplicados.sort_values(["num_etiquetas", "num_splits", "cantidad"], ascending=False).head(30)


Textos duplicados únicos: 41
Textos duplicados con etiquetas distintas (contradictorios): 3
Textos duplicados presentes en más de un split: 16


,text,cantidad,etiquetas,splits,num_etiquetas,num_splits
1,"Ahora, olvídese del asunto y déjelo a mi cargo",2,"[negative, neutral]","[train, validation]",2,2
30,Vamos al toque,2,"[neutral, positive]","[test, validation]",2,2
10,"El sol ☀️🌞 de shile es seco y quema la piel , en sol de Perú te arde la piel",2,"[negative, neutral]",[train],2,1
0,Acuérdate que todas son cachables,2,[neutral],"[test, train]",1,2
5,Cuando eres de cono pero te inspiras en hablar como pituco monse😂,2,[negative],"[train, validation]",1,2
7,El matón es su cachero,2,[negative],"[train, validation]",1,2
8,El patriotismo es la peor de las huachaferías,2,[negative],"[test, train]",1,2
11,Elijo el DORADO! <3 para nutrir mi cabello seco debido a los tintes y planchas 🙊 espero ganar!,2,[neutral],"[train, validation]",1,2
13,Es una huachafita con el pelo pintado,2,[negative],"[train, validation]",1,2
14,"Escúchame bien, serrano conchatumadre",2,[negative],"[test, train]",1,2


## Nulos o textos vacíos

In [18]:
for nombre, df in dfs.items():
    print(f"\n{nombre.upper()}")
    print(df.isnull().sum())
    vacios = df[df[columna_texto_principal].isna() | (df[columna_texto_principal].astype(str).str.strip() == "")]
    print("Textos vacíos:", len(vacios))


TRAIN
text          0
label         0
label_name    0
dtype: int64
Textos vacíos: 0

VALIDATION
text          0
label         0
label_name    0
dtype: int64
Textos vacíos: 0

TEST
text          0
label         0
label_name    0
dtype: int64
Textos vacíos: 0


## Patrones de texto a limpiar

In [19]:
patron_tildes_vocales = r"[áéíóúÁÉÍÓÚ]"
patron_url_real = r"(?:http\S+|www\.\S+|https?://\S+)"
patron_hashtag = r"#\w+"
patron_mencion = r"@\w+"
patron_repetidos = r"(.)\1{3,}"

for nombre, df in dfs.items():
    print(f"\n{nombre.upper()}")
    print("Textos con vocales con tilde:", df[columna_texto_principal].str.contains(patron_tildes_vocales, regex=True, na=False).sum())
    print("Textos con URLs reales:", df[columna_texto_principal].str.contains(patron_url_real, regex=True, case=False, na=False).sum())
    print("Textos con hashtags:", df[columna_texto_principal].str.contains(patron_hashtag, regex=True, na=False).sum())
    print("Textos con menciones:", df[columna_texto_principal].str.contains(patron_mencion, regex=True, na=False).sum())
    print("Textos con caracteres repetidos 4+ veces:", df[columna_texto_principal].str.contains(patron_repetidos, regex=True, na=False).sum())



TRAIN
Textos con vocales con tilde: 5065
Textos con URLs reales: 25
Textos con hashtags: 113
Textos con menciones: 37
Textos con caracteres repetidos 4+ veces: 1998

VALIDATION
Textos con vocales con tilde: 1562
Textos con URLs reales: 9
Textos con hashtags: 24
Textos con menciones: 11
Textos con caracteres repetidos 4+ veces: 626

TEST
Textos con vocales con tilde: 1301
Textos con URLs reales: 7
Textos con hashtags: 26
Textos con menciones: 8
Textos con caracteres repetidos 4+ veces: 490


/tmp/ipykernel_1152/1846590423.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print("Textos con caracteres repetidos 4+ veces:", df[columna_texto_principal].str.contains(patron_repetidos, regex=True, na=False).sum())


# Guardamos el pool completo de datos para el Notebook de limpieza

Guardamos df_all (que es los tres splits unidos, ocn la columna split original de hugging face).

In [20]:
df_all.to_csv("smps_hf_raw_pool.csv", index=False)
df_all.to_parquet("smps_hf_raw_pool.parquet", index=False)
print("Guardado. Total de filas en el pool:", len(df_all))


Guardado. Total de filas en el pool: 14589
